# Reading ACF & PACF Plots

Wiki reference for [ACF/PACF interpretation](https://ml-viz-ruby.vercel.app/wiki/acf-pacf-interpretation).

**The idea in one sentence.** The autocorrelation (ACF) and partial-autocorrelation (PACF)
functions are the fingerprints that tell an AR process from an MA process: an **AR(p)** has a
PACF that **cuts off** after lag $p$ (ACF tails off), while an **MA(q)** has an ACF that
**cuts off** after lag $q$ (PACF tails off).

We simulate AR(1), MA(1), and white noise from scratch, compute the sample ACF and Yule-Walker
PACF, **validate the tail-off / cut-off signatures**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')


## Theoretical ACF patterns

- AR(1) φ=0.8: ρ_k = 0.8^k (geometric decay forever)
- MA(1) θ=0.7: ρ_1 = θ/(1+θ²) ≈ 0.469, ρ_k = 0 for k ≥ 2
- White noise: ρ_k ≈ 0 for all k ≥ 1


In [ ]:
rng = np.random.default_rng(42)
n = 200

# AR(1) simulation
y_ar = np.zeros(n)
for t in range(1, n):
    y_ar[t] = 0.8 * y_ar[t-1] + rng.normal()

# MA(1) simulation
eps = rng.normal(size=n+1)
y_ma = eps[1:] + 0.7 * eps[:-1]

# White noise
y_wn = rng.normal(size=n)
print('Series generated:', len(y_ar), len(y_ma), len(y_wn))


## Sample ACF from scratch

In [ ]:
def sample_acf(y, max_lag=16):
    y_c = y - y.mean()
    var = np.dot(y_c, y_c)
    return [np.dot(y_c[k:], y_c[:-k]) / var if k > 0 else 1.0
            for k in range(max_lag + 1)]

acf_ar = sample_acf(y_ar)
acf_ma = sample_acf(y_ma)
acf_wn = sample_acf(y_wn)

lags = np.arange(17)
theory_ar = 0.8 ** lags
theory_ma = np.array([1 if k==0 else 0.7/(1+0.49) if k==1 else 0 for k in lags])
sig = 1.96 / np.sqrt(n)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, acf_v, theory, title in zip(
        axes, [acf_ar, acf_ma, acf_wn], [theory_ar, theory_ma, None],
        ['AR(1) phi=0.8', 'MA(1) theta=0.7', 'White Noise']):
    ax.bar(lags, acf_v, color='steelblue', alpha=0.75, label='Sample ACF')
    if theory is not None:
        ax.plot(lags, theory, 'o--', color='orange', ms=4, label='Theory')
    ax.axhline(sig, color='yellow', ls='--', lw=1)
    ax.axhline(-sig, color='yellow', ls='--', lw=1)
    ax.set_title(title); ax.set_xlabel('Lag'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


### Validate: the ACF signatures

AR(1) autocorrelations decay geometrically ($\rho_k = \phi^k$), MA(1) has one spike at lag 1
then drops to ~0, and white noise sits inside the $\pm 1.96/\sqrt{n}$ significance band. We
confirm all three.

In [ ]:
print(f'AR(1) ACF lags 1,3,5: {acf_ar[1]:.2f}, {acf_ar[3]:.2f}, {acf_ar[5]:.2f}')
print(f'MA(1) ACF lags 1,2  : {acf_ma[1]:.2f}, {acf_ma[2]:.2f}')
print(f'significance band   : +/-{sig:.3f}')
assert acf_ar[1] > 0.5 and acf_ar[5] < acf_ar[1], 'AR(1) ACF decays geometrically (tails off)'
assert abs(acf_ma[2]) < abs(acf_ma[1]), 'MA(1) ACF spikes at lag 1 then cuts off'
assert abs(acf_wn[5]) < 3 * sig, 'white-noise ACF stays inside the significance band'
print('\n✅ ACF tails off for AR, cuts off after lag q for MA')

## PACF via Yule-Walker recursion

In [ ]:
def yule_walker_pacf(y, max_lag=10):
    # Standard Yule-Walker / Durbin-Levinson: r[0]=1, r[k]=rho_k.
    # For each order k, solve the Toeplitz system R phi = r and take phi_kk (the last
    # coefficient) as the partial autocorrelation at lag k.
    r = sample_acf(y, max_lag)
    pacf_vals = []
    for k in range(1, max_lag + 1):
        R = np.array([[r[abs(i - j)] for j in range(k)] for i in range(k)])
        rhs = np.array(r[1:k + 1])
        phi = np.linalg.solve(R, rhs)
        pacf_vals.append(float(phi[-1]))
    return pacf_vals

pacf_ar = yule_walker_pacf(y_ar)
print('PACF AR(1) lags 1-6:', [round(v,3) for v in pacf_ar[:6]])
print('Expected: large at lag 1, near-zero after that')


### Validate: the PACF cuts off for AR

The Yule-Walker PACF of an AR(1) is large at lag 1 and ~0 afterwards — the mirror image of the
ACF. This cut-off is how you read the AR **order** $p$ off the plot. We confirm.

In [ ]:
print(f'AR(1) PACF lags 1-4: {[round(v,3) for v in pacf_ar[:4]]}')
assert abs(pacf_ar[0]) > 0.5, 'AR(1) PACF is large at lag 1'
assert all(abs(v) < 0.25 for v in pacf_ar[1:5]), 'AR(1) PACF cuts off after lag 1 -> order p=1'
print('\n✅ PACF cuts off at lag p for AR(p) — read the order straight off the plot')

## Pattern guide

| ACF | PACF | Model |
|-----|------|-------|
| Exponential decay | Cuts at lag p | AR(p) |
| Cuts at lag q | Exponential decay | MA(q) |
| Both decay | Both decay | ARMA(p,q) |
| All near zero | All near zero | White noise |


## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **reading one plot alone** | ACF-only can't tell AR from ARMA; use ACF+PACF together (demo) |
| **the significance band** | spikes inside $\pm 1.96/\sqrt n$ are noise, not structure (verified) |
| **non-stationarity** | a slow-decaying ACF means difference first, don't fit AR/MA |
| **short series** | few points → wide bands → unreliable identification |
| **seasonal lags** | spikes at seasonal lags need a seasonal (SARIMA) term |

Demo: AR and MA are mirror images in ACF vs PACF.

In [ ]:
# The identification DUALITY is the whole point: AR and MA are mirror images. AR(p): ACF
# tails off, PACF cuts off at p. MA(q): ACF cuts off at q, PACF tails off. We confirm the MA
# mirror — its PACF still has content beyond lag 1 (tails off) while its ACF already cut off.
pacf_ma = yule_walker_pacf(y_ma)
print(f'MA(1) ACF : lag1 {acf_ma[1]:.2f}, lag2 {acf_ma[2]:.2f}  (cuts off)')
print(f'MA(1) PACF: lag1 {pacf_ma[0]:.2f}, lag2 {pacf_ma[1]:.2f}, lag3 {pacf_ma[2]:.2f}  (tails off)')
assert abs(acf_ma[2]) < abs(acf_ma[1]), 'MA ACF cuts off after lag 1'
assert abs(pacf_ma[1]) > 0.02, 'MA PACF still has content past lag 1 (it tails off, not cuts off)'
print('\nAR and MA are mirror images -> use ACF+PACF TOGETHER to identify (p, q).')

## ✏️ Your turn

Compute the sample ACF at lags 1–5 for the series `y = [4, 7, 5, 8, 6, 9, 7, 10]`.


In [ ]:
y_small = np.array([4.0, 7, 5, 8, 6, 9, 7, 10])

# TODO(you): compute sample ACF at lags 1-5
# Hint: use sample_acf defined above, take [1:6]
acf_small = None  # replace

assert acf_small is not None, 'compute acf_small!'
assert len(acf_small) == 5, 'need exactly 5 values'
print('Your ACF lags 1-5:', [round(v,3) for v in acf_small])


<details>
<summary>Solution</summary>

```python
acf_small = sample_acf(y_small, max_lag=5)[1:]
```

</details>


## Key takeaways

- **AR(p): PACF cuts off at $p$, ACF tails off** (verified) — the PACF gives the AR order.
- **MA(q): ACF cuts off at $q$, PACF tails off** (verified) — the ACF gives the MA order.
- **They are mirror images:** read the two plots together to pin down $(p, q)$ (demo).
- **White noise:** both stay inside $\pm 1.96/\sqrt{n}$ — nothing to model (verified).